# CRM scanning: from single sites to clusters

In the PWM-scanning notebook we saw the **futility theorem** at work: a single CEBPA motif match on chr19 is almost never bound (only ~3% of strong matches were in a liver CEBPA peak). But we also saw a hint of hope: **regions with more CEBPA sites were bound more often**.

That is the idea behind **cis-regulatory module (CRM) scanning**. Enhancers are rarely a single binding site. They are usually clusters of sites, either for the same factor (**homotypic**) or for several factors that work together (**heterotypic**). Instead of scoring single sites, CRM scanners look for **dense clusters of motif matches**.

In this notebook we will

1. look at how **Hidden Markov Model (HMM)** based tools find clusters;
2. scan chr19 with **Cluster-Buster** using CEBPA alone (homotypic);
3. scan again with CEBPA plus four other liver transcription factors (heterotypic);
4. check how often the best clusters overlap a CEBPA peak, compared with random regions and scrambled motifs;
5. look at a well-known liver locus in the genome browser.

## 1. How do CRM scanners work?

Cluster-Buster (Frith et al. 2003) treats the DNA as a sequence produced by a simple model with two kinds of "states":

* a **background** state, which produces random DNA with the local base composition;
* **motif** states, one per matrix, which produce a binding site.

Inside a cluster, the model jumps between background and motif states; the **gap parameter** (`-g`, default 35 bp) is the expected distance between two neighbouring sites. Outside clusters, there is only background. For every stretch of DNA, Cluster-Buster asks: *how much more likely is this sequence if it is a cluster of motifs than if it is just background?* That log-likelihood ratio is the **cluster score**.

This has two advantages over counting sites above a threshold, like we did with windows in the previous notebook:

* weak sites still contribute a little, and strong sites contribute a lot: there is no hard threshold;
* the background is estimated **locally** (`-r`, default 100 bp on each side), so a GC-rich motif in a GC-rich region counts for less. Remember the GC difference we found around bound CEBPA sites!

Cluster-Buster is one of several tools from the same period:

| tool | idea |
|---|---|
| **Cister** (Frith et al. 2001) | HMM; gives for every position the probability that it lies in a cluster |
| **Ahab** (Rajewsky et al. 2002) | probabilistic model that also fits how strongly each factor contributes; used to predict *Drosophila* segmentation enhancers (Schroeder et al. 2004) |
| **Comet** (Frith et al. 2002) | cluster scores with a statistical significance (E-value) |
| **Cluster-Buster** (Frith et al. 2003) | fast successor of Cister and Comet; finds dense clusters of motif matches, genome-wide |

We use Cluster-Buster because it is fast, still maintained and easy to install.

## 2. Setup

Load the tools, and compile Cluster-Buster from its GitHub page (it is a small C++ program; `make` builds it in a few seconds).

In [2]:
mkdir -p crm_scanning
cd crm_scanning

In [3]:
module load MEME BEDTools SAMtools

In [4]:
git clone -q https://github.com/weng-lab/cluster-buster.git
cd cluster-buster
make > /dev/null 2>&1
cd ..
./cluster-buster/cbust -V

CLUSTER-BUSTER:  2024-09-27 07:33:36 -0400  commit: 06fee8b


## 3. Data

Like in the previous notebook, we work on chromosome 19.

In [5]:
samtools faidx ../genome/hg38.fa chr19 > chr19.fa
samtools faidx chr19.fa
cut -f 1,2 chr19.fa.fai > chr19.chrom.sizes

The CEBPA peaks on chr19 are our "true" set.

In [25]:
# grep -w chr19 ../res/CEBPA_peaks.filtered.narrowPeak | cut -f 1-3 > cebpa_peaks.bed
wc -l cebpa_peaks.bed

77 cebpa_peaks.bed


## 4. The motifs

Besides CEBPA, we use four transcription factors that are known to work together with CEBPA in liver enhancers:

| TF | JASPAR | consensus |
|---|---|---|
| CEBPA | MA0102.3 | ATTGCACAATA |
| HNF4A | MA0114.5 | CAAAGTCCA |
| FOXA2 | MA0047.4 | TGTAAACA |
| HNF1A | MA0046.3 | GTTAATGATTAAC |
| ONECUT1 (HNF6) | MA0679.3 | AAATCAATA |

Download them from JASPAR in MEME format.

In [26]:
for id in MA0102.3 MA0114.5 MA0047.4 MA0046.3 MA0679.3; do
  wget -q -O $id.meme https://jaspar.elixir.no/api/v1/matrix/$id.meme
done
ls *.meme

MA0046.3.meme  MA0047.4.meme  MA0102.3.meme  MA0114.5.meme  MA0679.3.meme


Cluster-Buster wants a simpler format: a line `>name`, followed by one row per position with the A, C, G and T values. We take the probability rows from the MEME file and multiply them by 100, so they look like counts (Cluster-Buster adds a small pseudocount to every value).

In [27]:
for id in MA0102.3 MA0114.5 MA0047.4 MA0046.3 MA0679.3; do
  awk -v id=$id '/letter-probability/ {m = 1; print ">" id; next} m && NF == 4 {print $1*100, $2*100, $3*100, $4*100; next} m {m = 0}' $id.meme > $id.cb
done
cat MA0102.3.cb

>MA0102.3
67.5741 8.6957 23.7303 0
0 0 0 100
0 0 0 100
4.6024 0 73.7955 21.602
0 100 0 0
78.052 8.317 6.9591 6.6719
17.3652 49.9674 0 32.6675
79.8146 20.1854 0 0
100 0 0 0
0 37.1328 11.777 51.0902
37.8901 32.3998 5.8102 23.9


We make two motif files:

* `cebpa.cb` with CEBPA only (**homotypic** clusters);
* `liver.cb` with all five factors (**heterotypic** clusters).

In [42]:
cp MA0046.3.cb cebpa.cb
cat MA0102.3.cb MA0114.5.cb MA0047.4.cb MA0046.3.cb MA0679.3.cb > liver.cb 
awk '/^>/' liver.cb

>MA0102.3
>MA0114.5
>MA0047.4
>MA0046.3
>MA0679.3


## 5. Homotypic clusters: CEBPA only

Run Cluster-Buster on chr19:

* `-f 5` writes a BED file;
* `-l` masks repeats (lowercase in the genome): tandem repeats can give very high but meaningless cluster scores;
* `-c 5` is the minimum cluster score (the default).

In [43]:
./cluster-buster/cbust -f 5 -l -c 5 cebpa.cb chr19.fa > cbust_cebpa_raw.bed
head -n 4 cbust_cebpa_raw.bed | cut -f 1-6,11

# chrom	genomic_start__bed	genomic_end__bed	cluster_id_or_motif_name	cluster_or_motif_score	strand	cluster_or_motif
chr19	53962453	53962676	chr19__cluster_1	18	+	cluster
chr19	53962453	53962466	MA0046.3	12	+	motif
chr19	53962453	53962466	MA0046.3	9.56	-	motif


The output has one line per **cluster** and, below it, one line per **motif** inside that cluster (column 11 says which). We keep the clusters, sorted from best to worst score (column 5).

In [44]:
awk 'BEGIN{OFS="\t"} $11 == "cluster" {print $1, $2, $3, $4, $5}' cbust_cebpa_raw.bed | sort -k5,5gr > cebpa_clusters.bed
wc -l cebpa_clusters.bed
head -n 5 cebpa_clusters.bed

266 cebpa_clusters.bed
chr19	53962453	53962676	chr19__cluster_1	18
chr19	6794006	6794230	chr19__cluster_2	17.6
chr19	33276162	33276175	chr19__cluster_3	14.9
chr19	35812890	35812953	chr19__cluster_4	14.7
chr19	6464464	6464477	chr19__cluster_5	12.2


### How many of the best clusters are in a CEBPA peak?

Take the top N clusters and count how many overlap a peak.

In [51]:
N=250
head -n $N cebpa_clusters.bed | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l

7


Is that more than chance? Move the same N clusters to random places on chr19 (same sizes) and count again.

In [52]:
for s in 1 2 3 4 5 6 7 8 9 10; do
  head -n $N cebpa_clusters.bed | bedtools shuffle -i - -g chr19.chrom.sizes -chrom -seed $s \
    | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l
done | awk '{s += $1} END {print "mean random overlap:", s/NR}'

mean random overlap: 0.3


Now for several values of N at once:

## 6. Heterotypic clusters: CEBPA + liver partners

Same run, now with all five liver factors.

In [53]:
./cluster-buster/cbust -f 5 -l -c 5 liver.cb chr19.fa > cbust_liver_raw.bed
awk 'BEGIN{OFS="\t"} $11 == "cluster" {print $1, $2, $3, $4, $5}' cbust_liver_raw.bed | sort -k5,5gr > liver_clusters.bed
wc -l liver_clusters.bed
head -n 5 liver_clusters.bed

922 liver_clusters.bed
chr19	796154	796690	chr19__cluster_1	19.1
chr19	33276162	33276216	chr19__cluster_2	18.9
chr19	58118972	58119213	chr19__cluster_3	14.5
chr19	56339274	56339863	chr19__cluster_4	14.3
chr19	53962439	53962676	chr19__cluster_5	13.9


In [54]:
head -n $N liver_clusters.bed | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l

7


In [55]:
for s in 1 2 3 4 5 6 7 8 9 10; do
  head -n $N liver_clusters.bed | bedtools shuffle -i - -g chr19.chrom.sizes -chrom -seed $s \
    | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l
done | awk '{s += $1} END {print "mean random overlap:", s/NR}'

mean random overlap: 0.3


### Which motifs make up the clusters?

For the top 100 liver clusters, which factors contribute to the clusters that are in a peak, and to those that are not? The motif lines in the raw output tell us which cluster each motif belongs to (column 12) and which motif it is (column 4).

In [56]:
head -n 500 liver_clusters.bed | bedtools intersect -u -a - -b cebpa_peaks.bed | cut -f 4 > top_bound_ids.txt
head -n 500 liver_clusters.bed | bedtools intersect -v -a - -b cebpa_peaks.bed | cut -f 4 > top_unbound_ids.txt
wc -l top_bound_ids.txt top_unbound_ids.txt

   9 top_bound_ids.txt
 491 top_unbound_ids.txt
 500 total


In [57]:
for set in top_bound top_unbound; do
  echo "== $set: motifs per cluster"
  awk 'NR==FNR {ids[$1] = 1; n++; next} $11 == "motif" && ($12 in ids) {c[$4]++}
       END {for (m in c) printf "%s\t%.2f\n", m, c[m]/n}' ${set}_ids.txt cbust_liver_raw.bed | sort
done

== top_bound: motifs per cluster
MA0046.3	1.22
MA0047.4	0.22
MA0102.3	0.67
MA0114.5	0.44
MA0679.3	0.56
== top_unbound: motifs per cluster
MA0046.3	0.34
MA0047.4	0.66
MA0102.3	0.43
MA0114.5	0.86
MA0679.3	0.36


| TF | JASPAR | consensus |
|---|---|---|
| CEBPA | MA0102.3 | ATTGCACAATA |
| HNF4A | MA0114.5 | CAAAGTCCA |
| FOXA2 | MA0047.4 | TGTAAACA |
| HNF1A | MA0046.3 | GTTAATGATTAAC |
| ONECUT1 (HNF6) | MA0679.3 | AAATCAATA |

## 7. Control: scrambled motifs

Shuffle the positions (rows) of every matrix. The information content stays the same, but the matrices are no longer real motifs. If the scrambled motifs find peaks just as well, our clusters are not about the motifs.

In [58]:
for id in MA0102.3 MA0114.5 MA0047.4 MA0046.3 MA0679.3; do
  head -n 1 $id.cb
  tail -n +2 $id.cb | shuf --random-source=<(yes 1)
done > liver_scrambled.cb
cat liver_scrambled.cb | head -n 13

>MA0102.3
78.052 8.317 6.9591 6.6719
67.5741 8.6957 23.7303 0
4.6024 0 73.7955 21.602
0 100 0 0
0 0 0 100
17.3652 49.9674 0 32.6675
0 37.1328 11.777 51.0902
100 0 0 0
0 0 0 100
37.8901 32.3998 5.8102 23.9
79.8146 20.1854 0 0
>MA0114.5


In [59]:
./cluster-buster/cbust -f 5 -l -c 5 liver_scrambled.cb chr19.fa > cbust_scrambled_raw.bed
awk 'BEGIN{OFS="\t"} $11 == "cluster" {print $1, $2, $3, $4, $5}' cbust_scrambled_raw.bed | sort -k5,5gr > scrambled_clusters.bed
wc -l scrambled_clusters.bed

589 scrambled_clusters.bed


In [60]:
echo -e "top_N\tliver\tscrambled"
for N in 25 50 100 250 500 1000; do
  real=$(head -n $N liver_clusters.bed | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l)
  scr=$(head -n $N scrambled_clusters.bed | bedtools intersect -u -a - -b cebpa_peaks.bed | wc -l)
  echo -e "$N\t$real\t$scr"
done

top_N	liver	scrambled


25	2	0
50	3	0
100	3	1
250	7	1
500	9	1
1000	13	1


So yeah, we didn't solve the futility theorem, but we came coser to understanding what enhancer regions look like!

### References

* Frith MC, Hansen U, Weng Z (2001) Detection of cis-element clusters in higher eukaryotic DNA. *Bioinformatics* 17: 878-889. (Cister)
* Frith MC, Spouge JL, Hansen U, Weng Z (2002) Statistical significance of clusters of motifs represented by position specific scoring matrices in nucleotide sequences. *Nucleic Acids Res* 30: 3214-3224. (Comet)
* Frith MC, Li MC, Weng Z (2003) Cluster-Buster: finding dense clusters of motifs in DNA sequences. *Nucleic Acids Res* 31: 3666-3668.
* Rajewsky N, Vergassola M, Gaul U, Siggia ED (2002) Computational detection of genomic cis-regulatory modules applied to body patterning in the early *Drosophila* embryo. *BMC Bioinformatics* 3: 30. (Ahab)
* Schroeder MD, et al. (2004) Transcriptional control in the segmentation gene network of *Drosophila*. *PLoS Biol* 2: e271.
* Aerts S, van Helden J, Sand O, Hassan BA (2007) Fine-tuning enhancer models to predict transcriptional targets across multiple genomes. *PLoS ONE* 2: e1115.

Most of these cells were generated by claude but reviewed by a human